# Visualización de Clases de agentes

In [2]:
pip install pylint graphviz

   ---------------------------------------- 0.0/522.3 kB ? eta -:--:--
   -- ------------------------------------- 30.7/522.3 kB ? eta -:--:--
   -------- ------------------------------- 112.6/522.3 kB 1.3 MB/s eta 0:00:01
   --------------------- ------------------ 286.7/522.3 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------  522.2/522.3 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 522.3/522.3 kB 2.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/47.1 kB ? eta -:--:--
   ---------------------------------------- 47.1/47.1 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/275.2 kB ? eta -:--:--
   ---------------------------------------- 275.2/275.2 kB 8.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/119.4 kB ? eta -:--:--
   ---------------------------------------- 119.4/119.4 kB 6.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/94.1 kB ? eta -:--:--
   --------------


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import subprocess
import os
import importlib
import sys
from typing import List, Optional

class ClassHierarchyAnalyzer:
    """Tool for analyzing and visualizing class hierarchies in Python libraries."""
    
    def __init__(self, output_dir: str = "class_diagrams"):
        """
        Initialize the analyzer.
        
        Args:
            output_dir (str): Directory where diagrams will be saved
        """
        self.output_dir = output_dir
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            
    def verify_module(self, module_name: str) -> bool:
        """
        Verify if a module can be imported.
        
        Args:
            module_name (str): Name of the module to verify
            
        Returns:
            bool: True if module can be imported, False otherwise
        """
        try:
            importlib.import_module(module_name)
            return True
        except ImportError as e:
            print(f"Error importing module {module_name}: {str(e)}")
            return False
    
    def get_module_path(self, module_name: str) -> Optional[str]:
        """
        Get the file path of a module.
        
        Args:
            module_name (str): Name of the module
            
        Returns:
            Optional[str]: Path to the module file or None if not found
        """
        try:
            module = importlib.import_module(module_name)
            if hasattr(module, '__file__'):
                return os.path.dirname(module.__file__)
            return None
        except ImportError:
            return None
    
    def generate_diagram(self, 
                        module_name: str,
                        specific_classes: Optional[List[str]] = None,
                        output_format: str = "png") -> Optional[str]:
        """
        Generate a class diagram for the specified module.
        
        Args:
            module_name (str): Name of the module to analyze (e.g., 'langchain.agents')
            specific_classes (List[str], optional): List of specific class names to analyze
            output_format (str): Output format ('png' or 'dot')
            
        Returns:
            Optional[str]: Path to the generated diagram or None if generation failed
        """
        # Verify module can be imported
        if not self.verify_module(module_name):
            print(f"Module {module_name} cannot be imported. Please check it's installed.")
            return None
            
        # Get module path
        module_path = self.get_module_path(module_name)
        if not module_path:
            print(f"Could not find path for module {module_name}")
            return None
            
        base_filename = module_name.replace(".", "_")
        output_path = os.path.join(self.output_dir, f"{base_filename}_classes")
        
        # Build pyreverse command
        cmd = [
            "pyreverse",
            "-o", output_format,
            "--project", base_filename,
            "--ignore", "tests",
            "--max-ancestors", "2",  # Limit ancestry depth for clearer diagrams
            "--all-ancestors",       # Show all inheritance relationships
        ]
        
        # Add specific classes if provided
        if specific_classes:
            for class_name in specific_classes:
                cmd.extend(["--class", class_name])
        
        # Add module path
        cmd.append(module_path)
        
        try:
            # Run pyreverse with more detailed output
            result = subprocess.run(
                cmd,
                check=True,
                capture_output=True,
                text=True,
                cwd=self.output_dir
            )
            
            # Print command output for debugging
            if result.stdout:
                print("Command output:", result.stdout)
            
            # Check for generated files
            expected_file = os.path.join(
                self.output_dir,
                f"classes_{base_filename}.{output_format}"
            )
            
            if os.path.exists(expected_file):
                new_path = f"{output_path}.{output_format}"
                os.rename(expected_file, new_path)
                print(f"Successfully generated diagram at: {new_path}")
                return new_path
            else:
                print(f"Expected output file not found: {expected_file}")
                return None
                
        except subprocess.CalledProcessError as e:
            print(f"Error running pyreverse command: {' '.join(cmd)}")
            print(f"Error output: {e.stderr}")
            return None
        except Exception as e:
            print(f"Unexpected error: {str(e)}")
            return None
    
    def compare_libraries(self,
                         modules: List[str],
                         output_format: str = "png") -> List[str]:
        """
        Generate comparative diagrams for multiple modules.
        
        Args:
            modules (List[str]): List of module names to compare
            output_format (str): Output format ('png' or 'dot')
            
        Returns:
            List[str]: Paths to generated diagrams
        """
        diagram_paths = []
        for module in modules:
            path = self.generate_diagram(module, output_format=output_format)
            if path:
                diagram_paths.append(path)
        return diagram_paths

# Example usage
# Ensure required packages are installed
required_packages = ["langchain", "crewai"]

import langchain
import crewai

for package in required_packages:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f"Package {package} not installed. Please install it with:")
        print(f"pip install {package}")

analyzer = ClassHierarchyAnalyzer()

# Example 1: Generate diagram for LangChain agents
print("\nGenerating LangChain agents diagram...")
langchain_diagram = analyzer.generate_diagram(
    "langchain.agents",
    specific_classes=["Agent", "AgentExecutor"]
)

# Example 2: Generate diagram for CrewAI agents
print("\nGenerating CrewAI agents diagram...")
crewai_diagram = analyzer.generate_diagram(
    "crewai.agents",
    specific_classes=["Agent"]
)

# Example 3: Compare both libraries
print("\nGenerating comparison diagrams...")
comparison_diagrams = analyzer.compare_libraries([
    "langchain.agents",
    "crewai.agents"
])

print("\nGenerated diagrams:", comparison_diagrams)




Generating LangChain agents diagram...
Error running pyreverse command: pyreverse -o png --project langchain_agents --ignore tests --max-ancestors 2 --all-ancestors --class Agent --class AgentExecutor c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain\agents
Error output: 

Generating CrewAI agents diagram...
Error running pyreverse command: pyreverse -o png --project crewai_agents --ignore tests --max-ancestors 2 --all-ancestors --class Agent c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\crewai\agents
Error output: 

Generating comparison diagrams...
Error running pyreverse command: pyreverse -o png --project langchain_agents --ignore tests --max-ancestors 2 --all-ancestors c:\Users\SoporteTI\Documents\project_directory\GitHubRepos\de_avatar_lowpoly\.venv\Lib\site-packages\langchain\agents
Error output: 
Error running pyreverse command: pyreverse -o png --project crewai_